In [ ]:
!pip install transformers datasets evaluate torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.4 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset

# Load GSM8K dataset
dataset = load_dataset("gsm8k", "main", split="train[:100]")  # specify the 'main' config



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [ ]:
questions = [item['question'] for item in dataset]
answers = [item['answer'] for item in dataset]

In [ ]:
def baseline_prompt(q):
    return f"Question: {q}\nAnswer:"

In [ ]:
def zero_shot_cot_prompt(q):
    return f"Question: {q}\nLet's think step by step.\nAnswer:"

In [ ]:
few_shot_examples = """
Example 1:
Q: If there are 2 oranges and you buy 3 more, how many oranges in total?
A: Step 1: Start with 2 oranges.
Step 2: Add 3 more oranges.
Step 3: Total oranges = 2 + 3 = 5.
Answer: 5 oranges.

Instructions for the next question: Explain your reasoning as if teaching someone. Break it into **numbered steps**, show all intermediate calculations, and finish with a final answer.

Now solve this:
Q: If there are 3 apples and you buy 2 more, how many apples in total?
A:
"""

def few_shot_cot_prompt(q):
    return f"{few_shot_examples}\nQuestion: {q}\nLet's think step by step.\nAnswer:"

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

model_name = "tiiuae/falcon-7b-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

generator = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=200)


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/281 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.word_embeddings.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:
prompt = "Question: If there are 3 apples and you buy 2 more, how many apples in total?\nLet's think step by step.\nAnswer:"
output = generator(prompt)[0]['generated_text']
print(output)

Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: If there are 3 apples and you buy 2 more, how many apples in total?
Let's think step by step.
Answer: There are 5 apples in total. (3 + 2 = 5)
